<div style="background: linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%); color: #ffffff; padding: 26px 30px; border-radius: 14px; font-family: 'Segoe UI', Roboto, Arial, sans-serif;">
<div style="font-size: 30px; font-weight: 700; line-height: 1.2; color: #ffffff;">QCC: the public frontier + an Atlas-powered chemistry refit that runs right here</div>
<div style="font-size: 15px; margin-top: 10px; color: #ffffff;">Google Verifiable Quantum Circuit Compilation &middot; tools from the QCC Circuit Atlas dataset &middot; every file re-verified with the unchanged official verifiers</div>
</div>

This version takes the best public file of each task (2026-11-15 morning) and continues the **chemistry weight refit
inside the notebook**, with the refit toolkit of the dataset
[QCC Circuit Atlas + Verified Refit Toolkit](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas)
(`tools/refit`, `tools/atlas_io.py`): the code is imported from the dataset, not copied here. **Add that dataset as an
input (or Copy & Edit this notebook, which keeps it attached)** and the whole pipeline runs for you.
The refit starts from the best public DF-THC weights, lowers `lambda_eff` at fixed angle words and keeps every
accuracy gate of the verifier with a margin of at least 1.5e-9 Ha. Loaders, words, invsqrt and DQI are unchanged
public files. If this notebook helps you, an upvote is appreciated.

| Task | File | Built by |
|---|---|---|
| invsqrt | `a4d46b17` | Leo Tang (T7421 line), carried by boristown and alex chilton; on the public 46-qubit line (manwithacat, Syed Asad Ali, Ali Haydar &Ouml;zda&#287;) |
| DQI | `821bf8d1` | Areeb Ali's 241 retained products on Syed Asad Ali's v12 decoder at extractor cutoff 18 (Ertu&#287;rul &Ouml;zer) |
| chemistry | maoxin123's v27 table, words and loaders `71735b23`; weights `56885f79` by boristown (gap-3 residue repair, C43 compiler), **refitted further here** | maoxin123, boristown, Atakan Aldemir, Heliosli, prvsiyan, Lera (marculera) and the authors they credit |

## Acknowledgements
Thank you to **Leo Tang**, **boristown**, **alex chilton**, **manwithacat (James Barlow)**, **maoxin123**, **Atakan Aldemir**,
**Syed Asad Ali**, **Ali Haydar &Ouml;zda&#287;**, **Areeb Ali**, **Ertu&#287;rul &Ouml;zer**, **Heliosli**, **prvsiyan**, **Lera (marculera)**,
**Kirill Tymchishin**, **SomeJoint**, **busyaprime**, **yoyobar**, **mrmeows**, the team **i want to run gemma 9b**,
**jday96314** and **Oleg Sher**; and Google Quantum AI and Kaggle for the competition, kickmix and the verifiers.

**What is ours here:** the chemistry refit toolkit (dataset) and its in-notebook run on the newest public weights.
Companion notebook: [QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).

<div style="background: #fff7e0; border-left: 6px solid #ffb703; padding: 12px 16px; border-radius: 8px; color: #1b1b1b;">
<b>Getting the submission file.</b> This notebook rebuilds, refits and verifies everything when it runs. To get
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">submission.zip</code>: <b>Copy &amp; Edit</b>, set
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">WRITE_SUBMISSION = True</code> in the next cell and <b>Run All</b>
(about 50 minutes on CPU). The dataset <b>QCC Circuit Atlas</b> must be attached (it is, after Copy &amp; Edit). Please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = False
REFIT_MINUTES = 10
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, math, csv, io, base64
import numpy as np
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
# Find every public file we need by its SHA-256, whatever notebook carries it.
WANT = {"invsqrt.kmb": "a4d46b17", "dqi.kmb": "821bf8d1", "dfthcsos/submission.npz": "56885f79"}
LOADERS = "71735b23"
got = {}
for z in inp.rglob("submission.zip"):
    try: Z = zipfile.ZipFile(z)
    except Exception: continue
    names = Z.namelist()
    for n, pre in WANT.items():
        if n in got or n not in names: continue
        b = Z.read(n)
        if sha(b).startswith(pre): got[n] = b
    if "loaders" not in got and "dfthcsos/angle_k_0.kmb" in names and sha(Z.read("dfthcsos/angle_k_0.kmb")).startswith(LOADERS):
        got["loaders"] = {n: Z.read(n) for n in names if n.startswith("dfthcsos/angle_k_")}
assert set(got) == set(WANT) | {"loaders"}, sorted(got)
for n in WANT: (SUB / n).write_bytes(got[n])
for n, b in got["loaders"].items(): (SUB / n).write_bytes(b)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in WANT}, "loaders", len(got["loaders"]))

## Our change: refit the chemistry weights here, with the Atlas toolkit
The refit code (our numpy port of prvsiyan's generalised reduced-gradient refit, Apache-2.0, with a full Newton
restoration step and Lera's gauge helpers) lives in the dataset **QCC Circuit Atlas**, folder `tools/refit`, and is
imported from there. It only changes `W` and `H2_I`; the verifier's own math (the competition's `dfthcsos.scoreinfra`)
checks every saved point exactly.

In [ ]:
import glob
ATLAS = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/circuits.csv", recursive=True)
              if os.path.isdir(os.path.join(os.path.dirname(p), "tools", "refit"))), None)
assert ATLAS, "Attach the dataset asteriospanoras/qcc-circuit-atlas (Add Input) and run again."
sys.path.insert(0, os.path.join(ATLAS, "tools"))
os.environ["QCC_VERIFIERS"] = str(VER); os.environ["QCC_CACHE"] = "/tmp/refitcache"
import atlas_io as A
from refit import chemeval as ce
from refit.model import Model, split, unsplit
from refit.grg2 import GRG2
ce.setup(verifiers=str(VER))
log("atlas tools from", ATLAS)
base = ce.load_npz(str(SUB / "dfthcsos/submission.npz"))
w0 = base["W"].astype(np.float64); wB0 = base["H2_I"].astype(np.float64)
def metrics(ww, bb): return ce.evaluate(base["angle_words"], ww, bb - ww.sum(1), base["tree_edges"])
m0 = metrics(w0, wB0); log("start", {k: m0[k] for k in ("lambda_eff", "error_1", "error_2", "E_gap")})
best = {"leff": m0["lambda_eff"], "W": w0, "H2_I": wB0}
deadline = time.time() + 60 * REFIT_MINUTES
class Stop(Exception): pass
def on_check(xe, okr):
    p, n, hp, hn = unsplit(xe, w0.shape); ww, bb = p - n, hp - hn
    m = metrics(ww, bb)
    ok = m["error_2"] < 0.02 - 1e-9 and m["error_1"] < 0.02 - 1e-9 and m["E_gap"] > 3.914898 + 1e-9 and m["gap_in_domain"] and m["Lambda"] > 49.0
    if ok and m["lambda_eff"] < best["leff"]:
        best.update(leff=m["lambda_eff"], W=ww.copy(), H2_I=bb.copy()); log("better", m["lambda_eff"])
    if time.time() > deadline: raise Stop()
M = Model(base["angle_words"], base["tree_edges"], *w0.shape[:2], cache_key="kaggle")
G = GRG2(M, w0.shape, log=lambda *a: None, m2=1.5e-9, mg=1.5e-9, m1=2e-9)
try: G.run(split(w0, wB0), 100000, on_check=on_check, check_every=50)
except Stop: pass
log("refit lambda_eff", m0["lambda_eff"], "->", best["leff"])
arrays = {k: np.array(base[k]) for k in base}
arrays["W"] = best["W"]; arrays["H2_I"] = best["H2_I"]
with open(SUB / "dfthcsos/submission.npz", "wb") as f: np.savez(f, **arrays)

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
r = {k: REF[k] / res[k]["score"] for k in REF}
print({k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP)

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
rows = A.circuits()
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
- **invsqrt** `a4d46b17`: [Leo Tang](https://www.kaggle.com/code/leotang09/vqcc-t7421-submission) (T7421 line), also carried by
  [boristown](https://www.kaggle.com/code/boristown/qcc-gap-3-residue-repair-and-c43-native-compiler) and
  [alex chilton](https://www.kaggle.com/code/alexchilton/27-31-lb-which-circuits-survive-private-salt); built on the 46-qubit line of
  [manwithacat (James Barlow)](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook),
  [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) and
  [Ali Haydar Özdağ](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package).
- **DQI** `821bf8d1`: [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-30687) on Syed Asad
  Ali's v12 decoder at extractor cutoff 18 ([Ertuğrul Özer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17)).
- **Chemistry**: [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry) (table, coded words,
  v27 loaders); weights `56885f79` by [boristown](https://www.kaggle.com/code/boristown/qcc-gap-3-residue-repair-and-c43-native-compiler)
  (on [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)'s, Heliosli's and earlier refits);
  [prvsiyan](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a)
  for the refit method (Apache-2.0); Lera (marculera)'s DF-THC line and the authors her NOTICE credits. **Ours:** the refit toolkit in the
  [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) and the refit run here.
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; this notebook's code is Apache-2.0.